In [ ]:
import numpy as np
from scipy.optimize import minimize

# Define the model functions
def calculate_electrical_cable_length_per_apartment(living_space, a, b):
    duct_length_apartment = 2 * (living_space ** 0.5) * a * b
    return duct_length_apartment

def calculate_electrical_cable_length(floor_height, total_duct_length_horizontal, num_floors):
    electrical_wire_length = (total_duct_length_horizontal + floor_height) * num_floors if num_floors > 0 else total_duct_length_horizontal
    return electrical_wire_length

def calculate_electrical_cable_weight(electrical_wire_length, density):
    weight = electrical_wire_length / 1000 * density
    return weight

# Define the objective function
def objective(params, validation_data, density):
    a, b = params
    total_error = 0
    for data in validation_data:
        living_space, floor_height, total_duct_length_horizontal, num_floors, actual_weight = data
        duct_length_apartment = calculate_electrical_cable_length_per_apartment(living_space, a, b)
        electrical_wire_length = calculate_electrical_cable_length(floor_height, duct_length_apartment, num_floors)
        predicted_weight = calculate_electrical_cable_weight(electrical_wire_length, density)
        total_error += (predicted_weight - actual_weight) ** 2
    return total_error / len(validation_data)

# Example validation data: (living_space, floor_height, total_duct_length_horizontal, num_floors, actual_weight)
validation_data = [
    (80, 3, 150, 10, 4000),
    (100, 3, 200, 12, 5000),
    # Add more data points as needed
]

# Initial parameter guesses
initial_params = [5, 1.25]
density = 4065  # Density of the wire material

# Perform the optimization
result = minimize(objective, initial_params, args=(validation_data, density), method='Nelder-Mead')
optimized_params = result.x

# Output the optimized parameters
print("Optimized Parameters:", optimized_params)

# Update the model functions with optimized parameters
optimized_a, optimized_b = optimized_params

# Use these optimized parameters in your model


# Monte Carlo

In [1]:
import numpy as np
import pandas as pd

# @ Radiator
# calculate the number of radiators needed for each apartment
def calculate_radiators_per_apartment(heated_area, num_rooms, num_kitchens):
    """
    Same I/O as your original function:
      inputs: heated_area (m2), num_rooms, num_kitchens
      output: int number of radiators

    Cross-validation idea:
      - n_room: layout / practice-based estimate (rooms + kitchens)
      - n_heat: coarse heat-demand-density estimate (W/m2 -> total W / W_per_radiator)
      - If they disagree strongly, allow a limited uplift above room-based,
        rather than blindly taking max or min.
    """

    # --- 1) Room/layout based (typical installation practice, good for stock) ---
    n_room = int(num_rooms) + int(num_kitchens)  # keep exactly your original rule basis

    # --- 2) Heat-based (coarse proxy; NOT full heat-loss model) ---
    heat_demand_density = 80   # W/m2 (your original value; consider 50–80 depending on stock)
    radiator_capacity = 1500   # W per radiator (depends on water temps; keep as your original)

    if heated_area is None or heated_area <= 0:
        n_heat = 0
    else:
        total_w = float(heated_area) * float(heat_demand_density)
        n_heat = int(np.ceil(total_w / float(radiator_capacity)))

    # --- 3) Cross-check + merge (single output) ---
    # Use room-based as baseline (stock realism), but prevent big underestimation:
    # allow heat-based to increase the count, capped to avoid blow-ups from uncertain assumptions.
    cap_extra = 2  # allow at most +2 radiators beyond n_room
    gap_threshold = 2  # if mismatch >=2, treat as notable disagreement

    if abs(n_heat - n_room) < gap_threshold:
        # close enough -> trust the simpler rule-based estimate
        n_final = n_room
    else:
        # disagreement -> take a conservative uplift, but cap it
        n_final = min(max(n_room, n_heat), n_room + cap_extra)

    # make sure it's at least 0
    return int(max(n_final, 0))

def perturb_continuous(x, sigma_rel, rng):
    # multiplicative noise, keep non-negative
    eps = rng.normal(0, sigma_rel)
    return max(x * (1 + eps), 0)

def perturb_integer(x, p0=0.70, p1=0.25, p2=0.05, rng=None):
    # error distribution: 0, ±1, ±2 (symmetric)
    r = rng.random()
    if r < p0:
        e = 0
    elif r < p0 + p1:
        e = 1
    else:
        e = 2
    sign = -1 if rng.random() < 0.5 else 1
    return int(max(x + sign * e, 0))

def perturb_kitchens(k, p_flip=0.10, rng=None):
    # simple flip model; adjust to your kitchen label space
    if rng.random() > p_flip:
        return int(k)
    # flip among {0,1,2} with simple rule
    if k == 0: return 1
    if k == 1: return 0
    return 1

def radiator_mass_for_building(row):
    # you already have these mappings elsewhere
    n = calculate_radiators_per_apartment(row["heated_area"], row["num_rooms"], row["num_kitchens"])
    unit_w = row["unit_weight"]  # attach from decade mapping beforehand
    return n * unit_w

def monte_carlo_sa(df_sample, n_mc=300,
                   sigma_area_rel=0.10,
                   room_err_probs=(0.7, 0.25, 0.05),
                   p_kitchen_flip=0.10,
                   seed=42):
    rng = np.random.default_rng(seed)
    totals = []

    for _ in range(n_mc):
        dfp = df_sample.copy()

        dfp["heated_area_p"] = dfp["heated_area"].apply(lambda x: perturb_continuous(x, sigma_area_rel, rng))
        dfp["num_rooms_p"] = dfp["num_rooms"].apply(lambda x: perturb_integer(x, *room_err_probs, rng=rng))
        dfp["num_kitchens_p"] = dfp["num_kitchens"].apply(lambda x: perturb_kitchens(x, p_flip=p_kitchen_flip, rng=rng))

        # run model
        masses = []
        for _, r in dfp.iterrows():
            n = calculate_radiators_per_apartment(r["heated_area_p"], r["num_rooms_p"], r["num_kitchens_p"])
            masses.append(n * r["unit_weight"])
        totals.append(np.sum(masses))

    totals = np.array(totals)
    out = {
        "mean_total_mass": float(totals.mean()),
        "median_total_mass": float(np.median(totals)),
        "p05": float(np.quantile(totals, 0.05)),
        "p95": float(np.quantile(totals, 0.95)),
        "cv": float(totals.std() / totals.mean())
    }
    return out, totals


In [3]:
import pandas as pd
import numpy as np

# -----------------------------
# 0) PATH
# -----------------------------
file_path = r"C:\Users\xiongs\OneDrive - ETH Zurich\Work\Code\mathematical predictive model\databases connection\df_apartment_agg_to_building.txt"

# -----------------------------
# 1) Your radiator rule (use your final version)
# -----------------------------
def calculate_radiators_per_apartment(heated_area, num_rooms, num_kitchens):
    n_room = int(num_rooms) + int(num_kitchens)

    heat_demand_density = 80   # W/m2
    radiator_capacity = 1500   # W

    if heated_area is None or heated_area <= 0:
        n_heat = 0
    else:
        total_w = float(heated_area) * float(heat_demand_density)
        n_heat = int(np.ceil(total_w / float(radiator_capacity)))

    cap_extra = 2
    gap_threshold = 2

    if abs(n_heat - n_room) < gap_threshold:
        n_final = n_room
    else:
        n_final = min(max(n_room, n_heat), n_room + cap_extra)

    return int(max(n_final, 0))

# -----------------------------
# 2) Radiator decade mapping
# -----------------------------
radiator_data = {
    '1930s and below': {'type': 'no radiator', 'material': 'cast iron', 'unit_weight': 0, 'lifespan': 20},
    '1940s': {'type': 'cast iron radiator', 'material': 'cast iron', 'unit_weight': 63, 'lifespan': 20},
    '1950s': {'type': 'steel panel radiator', 'material': 'steel', 'unit_weight': 21, 'lifespan': 50},
    '1960s': {'type': 'convector radiator', 'material': 'steel', 'unit_weight': 42, 'lifespan': 30},
    '1970s': {'type': 'column radiator', 'material': 'cast iron', 'unit_weight': 82, 'lifespan': 50},
    '1980s': {'type': 'steel panel radiator', 'material': 'steel', 'unit_weight': 21, 'lifespan': 50},
    '1990s': {'type': 'low surface temperature (LST) radiator', 'material': 'steel', 'unit_weight': 38, 'lifespan': 50},
    '2000s and beyond': {'type': 'low surface temperature (LST) radiator', 'material': 'steel', 'unit_weight': 38, 'lifespan': 50}
}

def year_to_decade(y):
    if pd.isna(y):
        return "2000s and beyond"
    y = float(y)
    if y < 1940: return "1930s and below"
    if y < 1950: return "1940s"
    if y < 1960: return "1950s"
    if y < 1970: return "1960s"
    if y < 1980: return "1970s"
    if y < 1990: return "1980s"
    if y < 2000: return "1990s"
    return "2000s and beyond"

# -----------------------------
# 3) Build df_sample (only needed columns)
# -----------------------------
df = pd.read_csv(file_path)

cols_needed = ["EGID", "GBAUJ", "GEBF", "WAREA", "WAZIM", "WKCHE"]
# baseline output column optional
if "number of radiators" in df.columns:
    cols_needed += ["number of radiators"]

df_sa = df[cols_needed].copy()

# heated area proxy: prefer GEBF, fallback WAREA
df_sa["heated_area"] = pd.to_numeric(df_sa["GEBF"], errors="coerce")
mask = df_sa["heated_area"].isna() | (df_sa["heated_area"] <= 0)
df_sa.loc[mask, "heated_area"] = pd.to_numeric(df_sa.loc[mask, "WAREA"], errors="coerce")

df_sa = df_sa.rename(columns={"WAZIM": "num_rooms", "WKCHE": "num_kitchens"})
df_sa["num_rooms"] = pd.to_numeric(df_sa["num_rooms"], errors="coerce")
df_sa["num_kitchens"] = pd.to_numeric(df_sa["num_kitchens"], errors="coerce").fillna(0)

df_sa["GBAUJ"] = pd.to_numeric(df_sa["GBAUJ"], errors="coerce")
df_sa["decade"] = df_sa["GBAUJ"].apply(year_to_decade)
df_sa["unit_weight"] = df_sa["decade"].map({k: v["unit_weight"] for k, v in radiator_data.items()})

# clean
df_sa = df_sa.dropna(subset=["heated_area", "num_rooms", "unit_weight"])
df_sa["num_rooms"] = df_sa["num_rooms"].astype(int)
df_sa["num_kitchens"] = df_sa["num_kitchens"].astype(int)

df_sample = df_sa[["EGID", "heated_area", "num_rooms", "num_kitchens", "unit_weight"]].copy()

# optional: sample to keep fast (adjust)
df_sample = df_sample.sample(n=min(20000, len(df_sample)), random_state=42)

# -----------------------------
# 4) Baseline mass
# -----------------------------
def total_mass(df_in):
    n_list = [
        calculate_radiators_per_apartment(r.heated_area, r.num_rooms, r.num_kitchens)
        for r in df_in.itertuples(index=False)
    ]
    n_arr = np.array(n_list, dtype=float)
    return float(np.sum(n_arr * df_in["unit_weight"].to_numpy()))

baseline_mass = total_mass(df_sample)

# -----------------------------
# 5) One-at-a-time (OAT) sensitivity
# -----------------------------
def oat_summary(df_in, baseline):
    out_rows = []

    # area perturbations
    for pct in [0.10, 0.20]:
        dfp = df_in.copy()
        dfp["heated_area"] = dfp["heated_area"] * (1 + pct)
        m = total_mass(dfp)
        out_rows.append(("heated_area", f"+{int(pct*100)}%", m, (m/baseline-1)*100))

        dfm = df_in.copy()
        dfm["heated_area"] = dfm["heated_area"] * (1 - pct)
        m2 = total_mass(dfm)
        out_rows.append(("heated_area", f"-{int(pct*100)}%", m2, (m2/baseline-1)*100))

    # rooms ±1
    for d in [+1, -1]:
        dfr = df_in.copy()
        dfr["num_rooms"] = np.maximum(dfr["num_rooms"] + d, 0)
        m = total_mass(dfr)
        out_rows.append(("num_rooms", f"{d:+d}", m, (m/baseline-1)*100))

    # kitchens flip 10% (simple)
    dfk = df_in.copy()
    rng = np.random.default_rng(42)
    flip = rng.random(len(dfk)) < 0.10
    # flip rule: 0<->1, 2->1
    nk = dfk["num_kitchens"].to_numpy()
    nk2 = nk.copy()
    nk2[flip & (nk == 0)] = 1
    nk2[flip & (nk == 1)] = 0
    nk2[flip & (nk >= 2)] = 1
    dfk["num_kitchens"] = nk2
    m = total_mass(dfk)
    out_rows.append(("num_kitchens", "10% flip", m, (m/baseline-1)*100))

    return pd.DataFrame(out_rows, columns=["input", "perturbation", "total_mass_kg", "change_vs_baseline_%"])

df_oat = oat_summary(df_sample, baseline_mass)

# -----------------------------
# 6) Lightweight Monte Carlo (P5–P95)
# -----------------------------
def perturb_integer(x, p0=0.70, p1=0.25, p2=0.05, rng=None):
    # error distribution: 0, ±1, ±2 (symmetric)
    r = rng.random()
    if r < p0:
        e = 0
    elif r < p0 + p1:
        e = 1
    else:
        e = 2
    sign = -1 if rng.random() < 0.5 else 1
    return int(max(int(x) + sign * e, 0))

def monte_carlo(df_in, n_mc=300, sigma_area_rel=0.10,
                room_err_probs=(0.7, 0.25, 0.05),
                p_kitchen_flip=0.10, seed=1):
    rng = np.random.default_rng(seed)
    totals = []

    base = df_in.copy()

    for _ in range(n_mc):
        dfp = base.copy()

        # area multiplicative noise
        eps = rng.normal(0, sigma_area_rel, size=len(dfp))
        dfp["heated_area"] = np.maximum(dfp["heated_area"].to_numpy() * (1 + eps), 0)

        # rooms integer noise
        p0, p1, p2 = room_err_probs
        dfp["num_rooms"] = [perturb_integer(v, p0, p1, p2, rng=rng) for v in dfp["num_rooms"].to_numpy()]

        # kitchens flip
        flip = rng.random(len(dfp)) < p_kitchen_flip
        nk = dfp["num_kitchens"].to_numpy()
        nk2 = nk.copy()
        nk2[flip & (nk == 0)] = 1
        nk2[flip & (nk == 1)] = 0
        nk2[flip & (nk >= 2)] = 1
        dfp["num_kitchens"] = nk2

        totals.append(total_mass(dfp))

    totals = np.array(totals, dtype=float)
    summary = {
        "baseline_mass_kg": baseline_mass,
        "mc_mean_kg": float(totals.mean()),
        "mc_p05_kg": float(np.quantile(totals, 0.05)),
        "mc_p95_kg": float(np.quantile(totals, 0.95)),
        "mc_range_%(p05-p95)": float((np.quantile(totals, 0.95) / np.quantile(totals, 0.05) - 1) * 100),
        "mc_cv": float(totals.std() / totals.mean())
    }
    return summary, totals

mc_summary, mc_totals = monte_carlo(df_sample, n_mc=300, sigma_area_rel=0.10)

# -----------------------------
# 7) Print results (what you paste into paper/rebuttal)
# -----------------------------
print("\n=== BASELINE total radiator mass (kg) ===")
print(baseline_mass)

print("\n=== OAT sensitivity (change vs baseline %) ===")
print(df_oat.sort_values("change_vs_baseline_%", key=lambda s: s.abs(), ascending=False))

print("\n=== Monte Carlo summary (P5–P95) ===")
print(mc_summary)


C:\Users\xiongs\AppData\Local\Temp\ipykernel_32864\358318294.py:64: DtypeWarning: Columns (6,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)



=== BASELINE total radiator mass (kg) ===
9706071.0

=== OAT sensitivity (change vs baseline %) ===
          input perturbation  total_mass_kg  change_vs_baseline_%
4     num_rooms           +1     10205752.0              5.148128
5     num_rooms           -1      9223452.0             -4.972342
3   heated_area         -20%      9313911.0             -4.040358
2   heated_area         +20%     10019917.0              3.233502
1   heated_area         -10%      9518547.0             -1.932028
0   heated_area         +10%      9887805.0              1.872375
6  num_kitchens     10% flip      9565726.0             -1.445951

=== Monte Carlo summary (P5–P95) ===
{'baseline_mass_kg': 9706071.0, 'mc_mean_kg': 9563346.326666666, 'mc_p05_kg': 9545752.7, 'mc_p95_kg': 9579615.7, 'mc_range_%(p05-p95)': 0.35474415757701383, 'mc_cv': 0.0010997759286490362}
